# Checkpoint A1: Cài đặt Ollama + Hermes + thiết lập môi trường

Chạy từng cell theo thứ tự để cài đặt và thiết lập toàn bộ môi trường.
Sau khi xong, tiếp tục từ **Bước A2: Khởi tạo 3 Agent Profiles**.

## A1.1 — Cài đặt Ollama

Chọn cell tương ứng với hệ điều hành của bạn.

In [1]:
# macOS: Cài Ollama qua Homebrew
# Bỏ comment dòng dưới nếu chưa cài
# !brew install ollama

# Linux / WSL2 Ubuntu: Cài Ollama qua script
# Bỏ comment dòng dưới nếu chưa cài
# !curl -fsSL https://ollama.com/install.sh | sh

# Window: sử dụng Windows Package Manager (`winget`) trong PowerShell:
#  ```powershell
#  winget install Ollama.Ollama
#  ```
# Sau khi cài đặt*: Ollama sẽ tự động chạy ẩn dưới khay hệ thống (System Tray).

import shutil
if shutil.which('ollama'):
    print('Ollama đã cài đặt.')
else:
    print('Chưa cài Ollama. Hãy chạy lệnh cài đặt trong Terminal của Antigravity hoặc PowerShell của Windows')

Ollama đã cài đặt.


In [2]:
# Khoi dong Ollama server chay nen
# Can mo terminal rieng chay: ollama serve

import subprocess, time

# Thu ket noi xem server da chay chua
import urllib.request
try:
    req = urllib.request.Request('http://localhost:11434')
    with urllib.request.urlopen(req, timeout=3) as resp:
        status = resp.read().decode()
    if 'Ollama is running' in status:
        print('Ollama server dang chay.')
    else:
        print(f'Phan hoi bat thuong: {status}')
except Exception:
    print('Ollama chua chay. Mo terminal moi va chay: ollama serve')
    print('Sau do chay lai cell nay.')

Ollama server dang chay.


## A1.2 — Tai mo hinh phu hop

Chon model phu hop voi cau hinh may.

In [3]:
# Chon model phu hop - bo comment model muon tai

# GPU + 16GB+ RAM:
# !ollama pull qwen3.5:7b-instruct

# Chi CPU + 16GB RAM:
# !ollama pull qwen3.5:3b

# Chi CPU + 8GB RAM:
# !ollama pull gemma4:e2b

# Chi CPU + 4GB RAM (mac dinh tai - sieu nhe cho may van phong):
!ollama pull qwen3.5:0.8b

print('Bo comment model muon tai o tren, sau do chay cell.')
print('Tai lan dau mat 3-5 phut.')

]11;?\

pulling manifest ⠋ 

pulling manifest ⠙ 

pulling manifest ⠹ 

pulling manifest ⠸ 

...

pulling manifest ⠏ 

pulling manifest ⠋ 

pulling manifest ⠙ 

pulling manifest 
pulling afb707b6b8fa: 100% ▕██████████████████▏ 1.0 GB                         
pulling 9be69ef46306: 100% ▕██████████████████▏  11 KB                         
pulling 9371364b27a5: 100% ▕██████████████████▏   65 B                         
pulling b14c6eab49f9: 100% ▕██████████████████▏  476 B                         
verifying sha256 digest 
writing manifest 
success 


Bo comment model muon tai o tren, sau do chay cell.
Tai lan dau mat 3-5 phut.


In [4]:
# Kiem tra model da co san
import subprocess, json, urllib.request

try:
    req = urllib.request.Request('http://localhost:11434/api/tags')
    with urllib.request.urlopen(req, timeout=5) as resp:
        data = json.loads(resp.read().decode())
        models = [m['name'] for m in data.get('models', [])]
    print(f'Ollama co {len(models)} model(s):')
    for m in models:
        print(f'  - {m}')
    if not models:
        print('Chua co model nao. Quay lai cell tren de tai.')
except Exception as e:
    print(f'Loi: {e}')

Ollama co 2 model(s):
  - qwen3.5:0.8b
  - qwen3.5:2b


## A1.3 — Cai dat Hermes Agent

In [5]:
# Cai Hermes Agent

# Windows (Native PowerShell):*
# pip install hermes-agent

# macOS (khuyến nghị — tránh lỗi PEP 668):*
# brew install hermes-agent

# Linux / WSL2:*
# pip install --user hermes-agent
# Hoặc dùng pipx (an toàn hơn pip)
# pipx install hermes-agent

import shutil
import subprocess
if shutil.which('hermes'):
    result = subprocess.run(['hermes', '--version'], capture_output=True, text=True)
    print(f'Hermes Agent: {result.stdout.strip()}')
else:
    print('Chưa cài Hermes. Chạy các lệnh cài đặt ở trên trong Terminal/Powershell.')

Hermes Agent: Hermes Agent v0.16.0 (2026.6.5)
Project: /opt/homebrew/Cellar/hermes-agent/2026.6.5/libexec/lib/python3.14/site-packages
Python: 3.14.5
OpenAI SDK: 2.24.0
Up to date


## A1.4 — Cau hinh Hermes ket noi Ollama

In [6]:
# Khoi tao Hermes (neu chua co config)
import os
import json
import urllib.request

config_dir = os.path.expanduser('~/.hermes')
config_path = os.path.join(config_dir, 'config.yaml')

os.makedirs(config_dir, exist_ok=True)

# Xac dinh model tu danh sach Ollama
model_name = 'qwen3.5:0.8b'  # mac dinh cho CPU / may van phong
try:
    req = urllib.request.Request('http://localhost:11434/api/tags')
    with urllib.request.urlopen(req, timeout=5) as resp:
        data = json.loads(resp.read().decode())
        models = [m['name'] for m in data.get('models', [])]
    if models:
        # Uu tien model tu cao xuong thap
        for preferred in ['qwen3.5:7b-instruct', 'qwen3.5:3b', 'gemma4:e2b', 'qwen3.5:1.5b-instruct', 'qwen3.5:0.8b']:
            if any(preferred in m for m in models):
                model_name = preferred
                break
        else:
            model_name = models[0]
except Exception:
    pass

config_content = f"""model:
  default: {model_name}
  provider: custom
  base_url: http://localhost:11434/v1
providers: {{}}
"""

with open(config_path, 'w') as f:
    f.write(config_content)

print(f'Da ghi config: {config_path}')
print(f'Model: {model_name}')

Da ghi config: /Users/shimazu/.hermes/config.yaml
Model: qwen3.5:0.8b


In [7]:
# Test Hermes chat nhanh
# Chu y: cell nay can hermes CLI, co the can chay thu cong
# hermes chat → go: "Ban dang dung mo hinh nao?" → /exit

import subprocess
try:
    result = subprocess.run(['hermes', 'chat', '--help'], 
                          capture_output=True, text=True, timeout=5)
    print('Hermes chat sẵn sàng.')
    print('Chạy thử trong terminal dòng lệnh: hermes chat')
except FileNotFoundError:
    print('Hermes chua cai. Quay lai buoc A1.3.')
except Exception as e:
    print(f'Loi: {e}')

Hermes chat sẵn sàng.
Chạy thử trong terminal dòng lệnh: hermes chat


## A1.5 — Khoi tao cau truc thu muc + tai lieu mo phong

In [8]:
# Tao cau truc thu muc thuc hanh
import os

base_dir = os.path.expanduser('../../output/vtn-session05')
subdirs = ['templates', 'runs', 'synthetic-data',
           'templates/skills', 'templates/skills/vtn-agent-skill',
           'templates/skills/vtn-agent-skill/kb',
           'templates/skills/vtn-agent-skill/scripts',
           'templates/skills/vtn-agent-skill/schemas']

for subdir in subdirs:
    full_path = os.path.join(base_dir, subdir)
    os.makedirs(full_path, exist_ok=True)

print(f'Da tao cau truc thu muc tai: {base_dir}')
for s in subdirs:
    print(f'  {s}/')

Da tao cau truc thu muc tai: ../../output/vtn-session05
  templates/
  runs/
  synthetic-data/
  templates/skills/
  templates/skills/vtn-agent-skill/
  templates/skills/vtn-agent-skill/kb/
  templates/skills/vtn-agent-skill/scripts/
  templates/skills/vtn-agent-skill/schemas/


In [9]:
# Tao thu muc mo phong /docs/simulated va /drafts
import os

# Neu chay WSL2/Linux va co sudo:
# os.makedirs('/docs/simulated', exist_ok=True)
# os.makedirs('/drafts', exist_ok=True)

# Tao tai lieu BGP mo phong
docs_dir = os.path.expanduser('../../output/vtn-session05/simulated-docs')
os.makedirs(docs_dir, exist_ok=True)

bgp_content = '''# Tai lieu mo phong: cau hinh BGP co ban tai VTN

## 1. Muc dich
Tai lieu nay mo ta khai niem co ban ve BGP va quy trinh cau hinh mo phong
danh cho bai lab dao tao. Khong dung cho he thong that.

## 2. BGP la gi
BGP (Border Gateway Protocol) la giao thuc dinh tuyen dung de trao doi
thong tin dinh tuyen giua cac he tu tri (Autonomous System) tren mang dien rong.

## 3. Quy trinh cau hinh BGP mo phong
1. Kiem tra trang thai router truoc thay doi.
2. Xac dinh so AS noi bo va AS lang gieng.
3. Khai bao tien trinh BGP mo phong.
4. Khai bao neighbor mo phong.
5. Kiem tra trang thai phien BGP.
6. Ghi log ket qua kiem tra.

## 4. Dieu kien dung
Neu phien BGP khong len trang thai Established trong thoi gian kiem thu,
dung thao tac va chuyen cho ky su van hanh bac 2.

## 5. Luu y an toan
Khong ap dung truc tiep noi dung nay len thiet bi that.
'''

bgp_path = os.path.join(docs_dir, 'vtn_bgp_config_sim.md')
with open(bgp_path, 'w', encoding='utf-8') as f:
    f.write(bgp_content)

print(f'Da tao tai lieu BGP: {bgp_path}')
print(f'Kich thuoc: {len(bgp_content)} bytes')

Da tao tai lieu BGP: ../../output/vtn-session05/simulated-docs/vtn_bgp_config_sim.md
Kich thuoc: 845 bytes


## A1.6 — Kiem tra tong ket

In [10]:
# Verify toan bo moi truong
import os, json, urllib.request, shutil

checks = []

# 1. Ollama server
try:
    req = urllib.request.Request('http://localhost:11434')
    with urllib.request.urlopen(req, timeout=3) as resp:
        status = resp.read().decode()
    checks.append(('Ollama server', 'Ollama is running' in status))
except Exception:
    checks.append(('Ollama server', False))

# 2. Model da tai
try:
    req = urllib.request.Request('http://localhost:11434/api/tags')
    with urllib.request.urlopen(req, timeout=5) as resp:
        data = json.loads(resp.read().decode())
        models = [m['name'] for m in data.get('models', [])]
    checks.append(('Model Ollama', len(models) > 0))
except Exception:
    checks.append(('Model Ollama', False))

# 3. Hermes CLI
checks.append(('Hermes CLI', shutil.which('hermes') is not None))

# 4. Hermes config
config_path = os.path.expanduser('~/.hermes/config.yaml')
checks.append(('Hermes config', os.path.exists(config_path)))

# 5. Thu muc thuc hanh
base_dir = os.path.expanduser('../../output/vtn-session05')
checks.append(('Thu muc ../../output/vtn-session05', os.path.isdir(base_dir)))

# 6. Tai lieu BGP
bgp_path = os.path.join(base_dir, 'simulated-docs/vtn_bgp_config_sim.md')
checks.append(('Tai lieu BGP mo phong', os.path.exists(bgp_path)))

print('=== KIEM TRA MOI TRUONG ===')
all_pass = True
for name, ok in checks:
    icon = 'PASS' if ok else 'FAIL'
    print(f'  [{icon}] {name}')
    if not ok:
        all_pass = False

if all_pass:
    print(f'\n Moi truong san sang. Tiep tu Bước A2.')
else:
    print(f'\n Con muc FAIL. Chay lai cac cell tuong ung o tren.')

=== KIEM TRA MOI TRUONG ===
  [PASS] Ollama server
  [PASS] Model Ollama
  [PASS] Hermes CLI
  [PASS] Hermes config
  [PASS] Thu muc ../../output/vtn-session05
  [PASS] Tai lieu BGP mo phong

 Moi truong san sang. Tiep tu Bước A2.


---

**Tiep tuc:** Quay lai `lab.md` → **Buoc A2: Khoi tao 3 Agent Profiles + thiet ke SOUL.md**